Audyt metodologii 2026-09-10: wyniki historyczne unieważnione. Definicje i ograniczenia: `../docs/methodology_audit.md`. Przeliczenia lokalne: `../data/processed/audit_v2/`.


target: Sprawdzić, czy w danych widać potencjał redukcji mocy/energii elektrofiltru bez pogorszenia emisji bazowej pyłu.


In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

DF_MODEL_PATH = DATA_PROCESSED / "df_model_clean_v1.parquet"
TAG_CLASSIFICATION_PATH = DATA_PROCESSED / "tag_classification_v1.xlsx"
RAPPING_FEATURES_PATH = DATA_PROCESSED / "rapping_features_v1.parquet"

print(PROJECT_ROOT)
print(DATA_PROCESSED)

In [ ]:
df = pd.read_parquet(DF_MODEL_PATH)
classified = pd.read_excel(TAG_CLASSIFICATION_PATH)
import sys
sys.path.insert(0, str(PROJECT_ROOT))
from src.time_analysis import (validate_time, time_shift, past_mean, lagged_corr,
    rapping_starts, rapping_features, complete_window, tail_mask as post_tail_mask, coverage)

validate_time(df)
rapping_features = rapping_features(df["008B05154"])

print("df:", df.shape)
print("classified:", classified.shape)
print("rapping_features:", rapping_features.shape)

print("df time range:", df.index.min(), "->", df.index.max())
print("rapping features time range:", rapping_features.index.min(), "->", rapping_features.index.max())

df.head()

In [ ]:
target_col = "008A01345"  # Pył BC1 Stężenie aktualne

esp_power_cols = [
    "008A02273",  # Zespół Zasil WN str 1A Moc Zespołu
    "008A02274",  # Zespół Zasil WN str 2A Moc Zespołu
    "008A02275",  # Zespół Zasil WN str 3A Moc Zespołu
]

classified[classified["tag"].isin(esp_power_cols + [target_col])][
    ["tag", "description", "symbol", "unit", "category"]
]

In [ ]:
df_energy = df.copy()

df_energy["esp_power_total"] = df_energy[esp_power_cols].sum(axis=1)

df_energy[
    esp_power_cols + ["esp_power_total", target_col]
].describe()

In [ ]:
critical_time_col = "minutes_since_rapping_3_collecting_start_0_5"

df_energy = df_energy.join(
    rapping_features[[critical_time_col]],
    how="left"
)

# poza oknem 0–5 min w rapping_features było -1
mask_after_rapping_0_3min = (
    (df_energy[critical_time_col] >= 0) &
    (df_energy[critical_time_col] <= 3)
)

df_energy_base = df_energy.loc[(~mask_after_rapping_0_3min) & df_energy[critical_time_col].notna()].copy()

print("Before:", df_energy.shape)
print("After removing 0–3 min after 008B05154:", df_energy_base.shape)
print("Removed samples:", mask_after_rapping_0_3min.sum())
print("Removed percent:", round(mask_after_rapping_0_3min.mean() * 100, 3), "%")

In [ ]:
plt.figure(figsize=(10, 5))

plt.hist(df_energy_base["esp_power_total"], bins=100)

plt.title("Distribution of total ESP power - base emission periods")
plt.xlabel("Total ESP power [kW]")
plt.ylabel("Number of samples")
plt.grid(True)
plt.show()

In [ ]:
power_thresholds = [40, 50, 75, 100, 150, 200]

for threshold in power_thresholds:
    n = (df_energy_base["esp_power_total"] > threshold).sum()
    p = (df_energy_base["esp_power_total"] > threshold).mean() * 100
    print(f"Power > {threshold} kW: {n} samples ({p:.3f}%)")

In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(
    df_energy_base["esp_power_total"],
    df_energy_base[target_col],
    s=3,
    alpha=0.2
)

plt.xlim(30, 60)

plt.title("Dust concentration vs total ESP power - base emission periods")
plt.xlabel("Total ESP power [kW]")
plt.ylabel("Dust concentration [mg/Nm3]")
plt.grid(True)
plt.show()

In [ ]:
power_bins = [30, 42, 52, 65, np.inf]
power_labels = ["low_power_30_42", "mid_power_42_52", "high_power_52_65", "extreme_power_gt_65"]

df_energy_base["power_regime"] = pd.cut(
    df_energy_base["esp_power_total"],
    bins=power_bins,
    labels=power_labels,
    include_lowest=True
)

power_regime_summary = (
    df_energy_base
    .groupby("power_regime", observed=True)
    .agg(
        n_samples=(target_col, "count"),
        mean_power=("esp_power_total", "mean"),
        median_power=("esp_power_total", "median"),
        mean_dust=(target_col, "mean"),
        median_dust=(target_col, "median"),
        q90_dust=(target_col, lambda x: x.quantile(0.90)),
        q95_dust=(target_col, lambda x: x.quantile(0.95)),
        max_dust=(target_col, "max"),
    )
    .reset_index()
)

power_regime_summary

In [ ]:
boiler_load_cols = [
    "016A00219",  # R1 P13 Moc czynna całkowita
    "016A00396",  # R1 P18 Moc czynna całkowita
]

df_energy_base["boiler_load_total"] = df_energy_base[boiler_load_cols].sum(axis=1)

power_regime_process_summary = (
    df_energy_base
    .groupby("power_regime", observed=True)
    .agg(
        n_samples=(target_col, "count"),
        mean_esp_power=("esp_power_total", "mean"),
        mean_dust=(target_col, "mean"),
        median_dust=(target_col, "median"),
        mean_boiler_load=("boiler_load_total", "mean"),
        median_boiler_load=("boiler_load_total", "median"),
    )
    .reset_index()
)

power_regime_process_summary

In [ ]:
df_energy_base["boiler_load_bin"] = pd.qcut(
    df_energy_base["boiler_load_total"],
    q=4,
    duplicates="drop"
)

boiler_power_summary = (
    df_energy_base
    .groupby(["boiler_load_bin", "power_regime"], observed=True)
    .agg(
        n_samples=(target_col, "count"),
        mean_boiler_load=("boiler_load_total", "mean"),
        mean_esp_power=("esp_power_total", "mean"),
        median_esp_power=("esp_power_total", "median"),
        mean_dust=(target_col, "mean"),
        median_dust=(target_col, "median"),
        q90_dust=(target_col, lambda x: x.quantile(0.90)),
        q95_dust=(target_col, lambda x: x.quantile(0.95)),
    )
    .reset_index()
)

boiler_power_summary

In [ ]:
boiler_power_summary_filtered = boiler_power_summary[
    boiler_power_summary["n_samples"] >= 200
].copy()

boiler_power_summary_filtered

In [ ]:
plt.figure(figsize=(10, 6))

for load_bin, group in boiler_power_summary_filtered.groupby("boiler_load_bin", observed=True):
    group = group.sort_values("mean_esp_power")

    plt.plot(
        group["mean_esp_power"],
        group["median_dust"],
        marker="o",
        label=str(load_bin)
    )

plt.title("Median dust vs ESP power by boiler load range")
plt.xlabel("Mean total ESP power [kW]")
plt.ylabel("Median dust concentration [mg/Nm3]")
plt.legend(title="Boiler load bin", fontsize=8)
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

for load_bin, group in boiler_power_summary_filtered.groupby("boiler_load_bin", observed=True):
    group = group.sort_values("mean_esp_power")

    plt.plot(
        group["mean_esp_power"],
        group["q95_dust"],
        marker="o",
        label=str(load_bin)
    )

plt.title("Q95 dust vs ESP power by boiler load range")
plt.xlabel("Mean total ESP power [kW]")
plt.ylabel("Q95 dust concentration [mg/Nm3]")
plt.legend(title="Boiler load bin", fontsize=8)
plt.grid(True)
plt.show()

In [ ]:
boiler_power_summary_filtered.sort_values(
    ["boiler_load_bin", "mean_esp_power"]
)

In [ ]:
boiler_power_summary_filtered = boiler_power_summary_filtered.copy()

boiler_power_summary_filtered["q95_per_kw"] = (
    boiler_power_summary_filtered["q95_dust"] /
    boiler_power_summary_filtered["mean_esp_power"]
)

boiler_power_summary_filtered["mean_dust_per_kw"] = (
    boiler_power_summary_filtered["mean_dust"] /
    boiler_power_summary_filtered["mean_esp_power"]
)

boiler_power_summary_filtered.sort_values(
    ["boiler_load_bin", "mean_esp_power"]
)

In [ ]:
tradeoff_rows = []

for load_bin, group in boiler_power_summary_filtered.groupby("boiler_load_bin", observed=True):
    group = group.sort_values("mean_esp_power").reset_index(drop=True)

    if len(group) < 2:
        continue

    ref = group.iloc[0]

    for i in range(1, len(group)):
        row = group.iloc[i]

        tradeoff_rows.append({
            "boiler_load_bin": load_bin,
            "reference_power": ref["mean_esp_power"],
            "compared_power": row["mean_esp_power"],
            "delta_power_kw": row["mean_esp_power"] - ref["mean_esp_power"],
            "reference_q95_dust": ref["q95_dust"],
            "compared_q95_dust": row["q95_dust"],
            "delta_q95_dust": row["q95_dust"] - ref["q95_dust"],
            "reference_median_dust": ref["median_dust"],
            "compared_median_dust": row["median_dust"],
            "delta_median_dust": row["median_dust"] - ref["median_dust"],
            "reference_n": ref["n_samples"],
            "compared_n": row["n_samples"],
        })

tradeoff_summary = pd.DataFrame(tradeoff_rows)

tradeoff_summary

In [ ]:
process_context_cols = [
    "016A00219",  # boiler load P13
    "016A00396",  # boiler load P18
    "008A00720",  # T spalin za podgrz. pow. OPP str.L
    "008A00728",  # T spalin za podgrz. pow. OPP str.P
    "008A01350",  # O2 lewa
    "008A00741",  # O2 przed OPP L
    "008A00742",  # O2 przed OPP P
]

classified[classified["tag"].isin(process_context_cols)][
    ["tag", "description", "symbol", "unit", "category"]
]

In [ ]:
df_energy_base["boiler_load_total"] = (
    df_energy_base["016A00219"] + df_energy_base["016A00396"]
)

df_energy_base["flue_gas_temp_mean"] = (
    df_energy_base[["008A00720", "008A00728"]].mean(axis=1)
)

df_energy_base["o2_before_esp_mean"] = (
    df_energy_base[["008A00741", "008A00742"]].mean(axis=1)
)

context_summary = df_energy_base[
    [
        "boiler_load_total",
        "flue_gas_temp_mean",
        "o2_before_esp_mean",
        "esp_power_total",
        target_col
    ]
].describe()

context_summary

In [ ]:
context_cols = [
    "boiler_load_total",
    "flue_gas_temp_mean",
    "o2_before_esp_mean"
]

typical_ranges = {}

for col in context_cols:
    q25 = df_energy_base[col].quantile(0.25)
    q75 = df_energy_base[col].quantile(0.75)
    typical_ranges[col] = (q25, q75)
    print(f"{col}: {q25:.3f} -> {q75:.3f}")

mask_typical_process = np.ones(len(df_energy_base), dtype=bool)

for col, (q25, q75) in typical_ranges.items():
    mask_typical_process &= (
        (df_energy_base[col] >= q25) &
        (df_energy_base[col] <= q75)
    )

df_energy_typical = df_energy_base.loc[mask_typical_process].copy()

print("Base emission samples:", len(df_energy_base))
print("Typical process samples:", len(df_energy_typical))
print("Percent retained:", round(len(df_energy_typical) / len(df_energy_base) * 100, 2), "%")

In [ ]:
plt.figure(figsize=(10, 5))

plt.hist(df_energy_typical["esp_power_total"], bins=80)

plt.title("Distribution of total ESP power - typical process conditions")
plt.xlabel("Total ESP power [kW]")
plt.ylabel("Number of samples")
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(
    df_energy_typical["esp_power_total"],
    df_energy_typical[target_col],
    s=3,
    alpha=0.25
)

plt.xlim(30, 80)

plt.title("Dust concentration vs total ESP power - typical process conditions")
plt.xlabel("Total ESP power [kW]")
plt.ylabel("Dust concentration [mg/Nm3]")
plt.grid(True)
plt.show()

In [ ]:
df_energy_typical["power_regime"] = pd.cut(
    df_energy_typical["esp_power_total"],
    bins=power_bins,
    labels=power_labels,
    include_lowest=True
)

typical_power_summary = (
    df_energy_typical
    .groupby("power_regime", observed=True)
    .agg(
        n_samples=(target_col, "count"),
        mean_power=("esp_power_total", "mean"),
        median_power=("esp_power_total", "median"),
        mean_dust=(target_col, "mean"),
        median_dust=(target_col, "median"),
        q90_dust=(target_col, lambda x: x.quantile(0.90)),
        q95_dust=(target_col, lambda x: x.quantile(0.95)),
        max_dust=(target_col, "max"),
        mean_boiler_load=("boiler_load_total", "mean"),
        mean_temp=("flue_gas_temp_mean", "mean"),
        mean_o2=("o2_before_esp_mean", "mean"),
    )
    .reset_index()
)

typical_power_summary

In [ ]:
plt.figure(figsize=(9, 5))

plt.plot(
    typical_power_summary["mean_power"],
    typical_power_summary["median_dust"],
    marker="o",
    label="Median dust"
)

plt.plot(
    typical_power_summary["mean_power"],
    typical_power_summary["q90_dust"],
    marker="o",
    label="Q90 dust"
)

plt.plot(
    typical_power_summary["mean_power"],
    typical_power_summary["q95_dust"],
    marker="o",
    label="Q95 dust"
)

plt.title("Dust vs ESP power under typical process conditions")
plt.xlabel("Mean total ESP power [kW]")
plt.ylabel("Dust concentration [mg/Nm3]")
plt.legend()
plt.grid(True)
plt.show()